# LumenY 8 — Separate Long / Short Direction Models (MFE-Filtered Signals)

Two independent binary classifiers trained **only on MFE Q50 > 70 bars**.

- **Long model**: target = `mfe_dir == 1` — predicts P(long is the better direction)
- **Short model**: target = `mfe_dir == 0` — predicts P(short is the better direction)

At inference: compute both scores, take the direction with highest confidence.
Each model learns its own feature patterns without shared directional bias.

**Key feature**: `mfe_q50_oof` (OOF prediction from MFE model) included as feature  
**Architecture**: two independent LightGBM binary classifiers  
**Saves**: `long_model.joblib` and `short_model.joblib`

In [1]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import matplotlib.pyplot as plt
import joblib
import warnings
import gc
warnings.filterwarnings('ignore')

from pathlib import Path
from sklearn.metrics import accuracy_score, log_loss, roc_auc_score, confusion_matrix

FEATURES_DIR  = Path('../backend/data/features_9')
MFE_MODEL_DIR = Path('../backend/models_9/mfe_q50')
MODELS_DIR    = Path('../backend/models_9/direction')
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# -- Training cutoff (same as MFE model) --
TRAIN_END = '2024-06-30'
MFE_THRESHOLD = 70.0  # filter: only train on bars where OOF mfe_q50 > 70

print('Ready.')
print(f'Separate Long/Short direction models on MFE Q50 > {MFE_THRESHOLD} population')
print(f'Training cutoff: {TRAIN_END}')

Ready.
Separate Long/Short direction models on MFE Q50 > 70.0 population
Training cutoff: 2024-06-30


## 1. Load Dataset

Load features_9. Build `mfe_dir` target (which direction traveled furthest over 72H).

In [2]:
# Load features_9 (all 15 pairs)
dfs = []
for f in sorted(FEATURES_DIR.glob('*_features.parquet')):
    tmp = pd.read_parquet(f)
    dfs.append(tmp)

df_all = pd.concat(dfs).sort_index()
del dfs
print(f'features_9: {df_all.shape}')
print(f'Pairs: {sorted(df_all["pair"].unique())}')
print(f'Date range: {df_all.index.min().date()} to {df_all.index.max().date()}')

features_9: (1518020, 327)
Pairs: ['AUDJPY', 'AUDNZD', 'AUDUSD', 'CADJPY', 'CHFJPY', 'EURAUD', 'EURGBP', 'EURJPY', 'EURUSD', 'GBPJPY', 'GBPUSD', 'NZDUSD', 'USDCAD', 'USDCHF', 'USDJPY']
Date range: 2009-09-25 to 2025-12-31


In [3]:
# Build feature cols (same exclusions as MFE model)
label_cols   = [c for c in df_all.columns if c.startswith('label_')] + [
    'mfe_long_pips', 'mfe_short_pips', 'mfe_atr_24', 'trail_long_bars',
    'trail_short_bars', 'trail_stop_pips'
]
drop_cols    = label_cols + ['pair']
feature_cols = [c for c in df_all.columns if c not in drop_cols]

print(f'Feature columns: {len(feature_cols)}')

# Create mfe_dir target: 1=long won, 0=short won
df_all['mfe_max'] = df_all[['mfe_long_pips', 'mfe_short_pips']].max(axis=1)
df_all['mfe_dir'] = (df_all['mfe_long_pips'] > df_all['mfe_short_pips']).astype(int)

print(f'mfe_dir distribution (all bars):')
print(df_all['mfe_dir'].value_counts())
print(f'Balance (1=long): {df_all["mfe_dir"].mean():.3f}')

# Split train/test
df_train = df_all[df_all.index <= TRAIN_END].copy()
df_test  = df_all[df_all.index > TRAIN_END].copy()
print(f'Train: {len(df_train):,} | Test: {len(df_test):,}')

Feature columns: 320
mfe_dir distribution (all bars):
mfe_dir
0    813787
1    704233
Name: count, dtype: int64
Balance (1=long): 0.464
Train: 1,378,758 | Test: 139,262


## 2. MFE Model OOF Predictions (Walk-Forward)

Re-run walk-forward CV to generate unbiased `mfe_q50_oof` for the training set.
These OOF preds are then used as a feature for the direction model — no leakage.

In [4]:
# Load MFE model bundle
mfe_bundle       = joblib.load(MFE_MODEL_DIR / 'model_1H_Q50.joblib')
mfe_feature_cols = mfe_bundle['feature_cols']
mfe_n_iters      = mfe_bundle['n_iters']

print(f'MFE model: {len(mfe_feature_cols)} features, {mfe_n_iters} iterations')

MFE model: 320 features, 423 iterations


In [5]:
def walk_forward_splits(n, n_splits=5, test_ratio=0.1):
    test_size = int(n * test_ratio)
    splits = []
    for i in range(n_splits):
        test_start = int(n * 0.5) + i * (int(n * 0.5) // n_splits)
        test_end   = test_start + test_size
        if test_end > n:
            break
        splits.append((list(range(0, test_start)), list(range(test_start, test_end))))
    return splits

y_mfe_all  = df_train['mfe_max']
valid_mask = y_mfe_all.notna()
X_mfe = df_train[mfe_feature_cols][valid_mask].ffill().fillna(0)
y_mfe = y_mfe_all[valid_mask]

splits      = walk_forward_splits(len(X_mfe))
oof_mfe_q50 = np.full(len(X_mfe), np.nan)

mfe_params = {
    'objective': 'quantile', 'alpha': 0.50, 'metric': 'quantile',
    'boosting_type': 'gbdt', 'n_estimators': mfe_n_iters,
    'learning_rate': 0.02, 'num_leaves': 64, 'max_depth': 6,
    'min_child_samples': 50, 'feature_fraction': 0.7,
    'bagging_fraction': 0.8, 'bagging_freq': 5,
    'reg_alpha': 0.1, 'reg_lambda': 0.1,
    'random_state': 42, 'n_jobs': -1, 'verbose': -1, 'device': 'gpu',
}

print(f'Generating MFE OOF predictions ({len(splits)} folds)...')
for fold, (train_idx, test_idx) in enumerate(splits):
    X_tr, y_tr = X_mfe.iloc[train_idx], y_mfe.iloc[train_idx]
    X_te = X_mfe.iloc[test_idx]
    model = lgb.LGBMRegressor(**mfe_params)
    model.fit(X_tr, y_tr, callbacks=[lgb.log_evaluation(-1)])
    oof_mfe_q50[test_idx] = model.predict(X_te)
    print(f'  Fold {fold+1}/{len(splits)}: {len(test_idx):,} samples')
    del model; gc.collect()

# Attach OOF predictions to training rows
df_train_valid = df_train[valid_mask].copy()
df_train_valid['mfe_q50_oof'] = oof_mfe_q50

# Test set: use final MFE model directly
X_test_mfe = df_test[mfe_feature_cols].ffill().fillna(0)
df_test['mfe_q50_oof'] = mfe_bundle['model'].predict(X_test_mfe)

valid_oof_count = (~np.isnan(oof_mfe_q50)).sum()
print(f'OOF coverage: {valid_oof_count:,} / {len(oof_mfe_q50):,}')
print(f'OOF mfe_q50 mean: {np.nanmean(oof_mfe_q50):.2f}')

Generating MFE OOF predictions (5 folds)...
  Fold 1/5: 133,515 samples
  Fold 2/5: 133,515 samples
  Fold 3/5: 133,515 samples
  Fold 4/5: 133,515 samples
  Fold 5/5: 133,515 samples
OOF coverage: 667,575 / 1,335,153
OOF mfe_q50 mean: 34.53


## 3. Filter to MFE Q50 > 70 Population

Only train direction model on bars where MFE signal is strong.
This mirrors the simulation gate exactly.

In [6]:
# Filter: only MFE Q50 > 70 bars
df_dir_train = df_train_valid[df_train_valid['mfe_q50_oof'] > MFE_THRESHOLD].copy()
df_dir_test  = df_test[df_test['mfe_q50_oof'] > MFE_THRESHOLD].copy()

print(f'=== MFE-Filtered Population ===')
print(f'Train: {len(df_dir_train):,} bars ({len(df_dir_train)/len(df_train_valid)*100:.1f}% of train)')
print(f'Test:  {len(df_dir_test):,} bars ({len(df_dir_test)/len(df_test)*100:.1f}% of test)')
print(f'')
print(f'Train mfe_dir balance: {df_dir_train["mfe_dir"].mean():.3f}  (long={df_dir_train["mfe_dir"].sum():,}  short={(df_dir_train["mfe_dir"]==0).sum():,})')
print(f'Test  mfe_dir balance: {df_dir_test["mfe_dir"].mean():.3f}   (long={df_dir_test["mfe_dir"].sum():,}  short={(df_dir_test["mfe_dir"]==0).sum():,})')
print(f'')
print(f'Long model  target: mfe_dir==1  (P(long is better direction))')
print(f'Short model target: mfe_dir==0  (P(short is better direction))')

=== MFE-Filtered Population ===
Train: 29,394 bars (2.2% of train)
Test:  16,207 bars (11.6% of test)

Train mfe_dir balance: 0.448  (long=13,159  short=16,235)
Test  mfe_dir balance: 0.364   (long=5,898  short=10,309)

Long model  target: mfe_dir==1  (P(long is better direction))
Short model target: mfe_dir==0  (P(short is better direction))


## 4. Walk-Forward CV — Long Model & Short Model (Separate)

In [7]:
def train_direction_model(df_train_filtered, df_test_filtered, target_col, model_name,
                          dir_feature_cols, n_splits=5):
    """
    Train a binary classifier for one direction (long or short).
    target_col: binary column (1=favorable, 0=not) already attached to df_train_filtered.
    Returns: (final_model, oof_probs, test_probs, best_avg_iter)
    """
    base_params = {
        'objective': 'binary', 'metric': 'binary_logloss',
        'boosting_type': 'gbdt', 'n_estimators': 3000,
        'learning_rate': 0.02, 'num_leaves': 64, 'max_depth': 6,
        'min_child_samples': 30, 'feature_fraction': 0.7,
        'bagging_fraction': 0.8, 'bagging_freq': 5,
        'reg_alpha': 0.1, 'reg_lambda': 0.1,
        'random_state': 42, 'n_jobs': -1, 'verbose': -1, 'device': 'gpu',
    }

    y_all    = df_train_filtered[target_col]
    valid    = y_all.notna()
    X        = df_train_filtered[dir_feature_cols][valid].ffill().fillna(0)
    y        = y_all[valid]
    splits   = walk_forward_splits(len(X), n_splits=n_splits)
    oof_prob = np.full(len(X), np.nan)
    best_iters = []

    print(f'\n--- {model_name} ---')
    print(f'  Samples: {len(X):,}  |  Target balance (1): {y.mean():.3f}  |  Folds: {len(splits)}')
    for fold, (train_idx, test_idx) in enumerate(splits):
        X_tr, y_tr = X.iloc[train_idx], y.iloc[train_idx]
        X_te, y_te = X.iloc[test_idx],  y.iloc[test_idx]
        m = lgb.LGBMClassifier(**base_params)
        m.fit(X_tr, y_tr,
              eval_set=[(X_te, y_te)],
              callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)])
        oof_prob[test_idx] = m.predict_proba(X_te)[:, 1]
        best_iters.append(m.best_iteration_)
        valid_fold = ~np.isnan(oof_prob[test_idx])
        acc_f = accuracy_score(y_te.values[valid_fold],
                               (oof_prob[test_idx][valid_fold] > 0.5).astype(int))
        print(f'  Fold {fold+1}: acc={acc_f:.4f}, best_iter={m.best_iteration_}')
        del m; gc.collect()

    avg_iter = max(50, int(np.mean(best_iters)))
    print(f'  Avg best iter: {avg_iter}')

    # Final model on all training data
    params_final = {**base_params, 'n_estimators': avg_iter}
    final_model  = lgb.LGBMClassifier(**params_final)
    final_model.fit(X, y, callbacks=[lgb.log_evaluation(-1)])

    # Test predictions
    X_test   = df_test_filtered[dir_feature_cols].ffill().fillna(0)
    y_test   = df_test_filtered[target_col]
    valid_t  = y_test.notna()
    test_prob = final_model.predict_proba(X_test[valid_t])[:, 1]

    return final_model, oof_prob, test_prob, avg_iter, valid, valid_t, y, y_test[valid_t]


# Feature set: features_9 columns + mfe_q50_oof
dir_feature_cols = feature_cols + ['mfe_q50_oof']
print(f'Direction feature cols: {len(dir_feature_cols)}')

Direction feature cols: 321


## 5. Train Both Models

In [ ]:
# Add binary targets to BOTH train and test
df_dir_train['target_long']  = df_dir_train['mfe_dir']
df_dir_train['target_short'] = (df_dir_train['mfe_dir'] == 0).astype(int)
df_dir_test['target_long']   = df_dir_test['mfe_dir']
df_dir_test['target_short']  = (df_dir_test['mfe_dir'] == 0).astype(int)

# Train LONG model
(long_model, oof_long_prob, test_long_prob,
 long_iters, valid_long, valid_long_t,
 y_long_train, y_long_test) = train_direction_model(
    df_dir_train, df_dir_test, 'target_long', 'LONG MODEL', dir_feature_cols)

# Train SHORT model
(short_model, oof_short_prob, test_short_prob,
 short_iters, valid_short, valid_short_t,
 y_short_train, y_short_test) = train_direction_model(
    df_dir_train, df_dir_test, 'target_short', 'SHORT MODEL', dir_feature_cols)

print('\nBoth models trained.')


--- LONG MODEL ---
  Samples: 29,394  |  Target balance (1): 0.448  |  Folds: 5
  Fold 1: acc=0.6108, best_iter=110
  Fold 2: acc=0.6339, best_iter=281
  Fold 3: acc=0.5886, best_iter=159
  Fold 4: acc=0.6512, best_iter=224
  Fold 5: acc=0.6683, best_iter=153
  Avg best iter: 185


KeyError: 'target_long'

## 6. OOF Evaluation — Long & Short Models

In [ ]:
def eval_oof(oof_prob, y_true, model_name):
    valid = ~np.isnan(oof_prob)
    p   = oof_prob[valid]
    y   = y_true.values[valid]
    acc = accuracy_score(y, (p > 0.5).astype(int))
    auc = roc_auc_score(y, p)
    print(f'{model_name}: OOF acc={acc:.4f}  AUC={auc:.4f}  n={valid.sum():,}')
    return acc, auc

print('=== OOF Evaluation ===')
acc_long_oof,  auc_long_oof  = eval_oof(oof_long_prob,  y_long_train,  'Long model ')
acc_short_oof, auc_short_oof = eval_oof(oof_short_prob, y_short_train, 'Short model')

# Combined OOF: direction = argmax(long_prob, short_prob)
# Align indices: both models trained on same filtered population
valid_both = (~np.isnan(oof_long_prob)) & (~np.isnan(oof_short_prob))
p_long  = oof_long_prob[valid_both]
p_short = oof_short_prob[valid_both]
y_actual = y_long_train.values[valid_both]  # mfe_dir: 1=long won

pred_combined = np.where(p_long >= p_short, 1, 0)
acc_combined  = accuracy_score(y_actual, pred_combined)

print(f'\nCombined (argmax):  OOF acc={acc_combined:.4f}  n={valid_both.sum():,}')
print(f'  Long predicted:  {pred_combined.sum():,} ({pred_combined.mean():.1%})')
print(f'  Short predicted: {(pred_combined==0).sum():,} ({(pred_combined==0).mean():.1%})')
print(f'  Actual long rate: {y_actual.mean():.3f}')

In [ ]:
print('=== Test Set Evaluation ===')

# Individual model accuracy
y_test_dir = df_dir_test.loc[valid_long_t, 'mfe_dir']
acc_long_test  = accuracy_score(y_long_test, (test_long_prob  > 0.5).astype(int))
acc_short_test = accuracy_score(y_short_test, (test_short_prob > 0.5).astype(int))
auc_long_test  = roc_auc_score(y_long_test,  test_long_prob)
auc_short_test = roc_auc_score(y_short_test, test_short_prob)

print(f'Long  model: acc={acc_long_test:.4f}  AUC={auc_long_test:.4f}  n={len(y_long_test):,}')
print(f'Short model: acc={acc_short_test:.4f}  AUC={auc_short_test:.4f}  n={len(y_short_test):,}')

# Combined: argmax
pred_test_combined = np.where(test_long_prob >= test_short_prob, 1, 0)
acc_test_combined  = accuracy_score(y_long_test, pred_test_combined)
print(f'\nCombined (argmax):  acc={acc_test_combined:.4f}  n={len(y_long_test):,}')
print(f'  Long predicted:   {pred_test_combined.sum():,} ({pred_test_combined.mean():.1%})')
print(f'  Short predicted:  {(pred_test_combined==0).sum():,} ({(pred_test_combined==0).mean():.1%})')
print(f'  Actual long rate: {y_long_test.mean():.3f}')

# Confidence threshold sweep
print(f'\n=== Confidence Threshold Sweep (Test) ===')
print(f'{"Min conf":>10} | {"N signals":>10} | {"Coverage":>9} | {"Acc":>7} | {"Long%":>7}')
print('-' * 55)
for thr in [0.50, 0.52, 0.55, 0.58, 0.60, 0.65]:
    conf   = np.maximum(test_long_prob, test_short_prob)
    mask   = conf >= thr
    if mask.sum() < 30:
        continue
    preds  = np.where(test_long_prob[mask] >= test_short_prob[mask], 1, 0)
    acc_t  = accuracy_score(y_long_test.values[mask], preds)
    long_r = preds.mean()
    print(f'{thr:>10.2f} | {mask.sum():>10,} | {mask.mean():>8.1%} | {acc_t:>7.4f} | {long_r:>7.1%}')

## 7. Save Models

In [ ]:
common_meta = {
    'feature_cols':     dir_feature_cols,
    'mfe_feature_cols': mfe_feature_cols,
    'train_end':        TRAIN_END,
    'mfe_threshold':    MFE_THRESHOLD,
    'task':             'direction_binary',
}

long_path  = MODELS_DIR / 'long_model.joblib'
short_path = MODELS_DIR / 'short_model.joblib'

joblib.dump({**common_meta, 'model': long_model,  'direction': 'long',  'n_iters': long_iters},  long_path)
joblib.dump({**common_meta, 'model': short_model, 'direction': 'short', 'n_iters': short_iters}, short_path)

print(f'Saved long  model ({long_path.stat().st_size/1024/1024:.1f} MB) -> {long_path}')
print(f'Saved short model ({short_path.stat().st_size/1024/1024:.1f} MB) -> {short_path}')

## 8. Per-Pair Breakdown (Test Set)

In [ ]:
results = pd.DataFrame({
    'actual_dir':  y_long_test.values,
    'prob_long':   test_long_prob,
    'prob_short':  test_short_prob,
    'pred_dir':    pred_test_combined,
    'pair':        df_dir_test.loc[valid_long_t, 'pair'].values,
    'mfe_q50':     df_dir_test.loc[valid_long_t, 'mfe_q50_oof'].values,
    'confidence':  np.maximum(test_long_prob, test_short_prob),
}, index=y_long_test.index)

print(f'=== Per-Pair Breakdown (Test Set) ===')
print(f'{"Pair":>8} | {"N":>6} | {"Acc":>7} | {"Long%pred":>10} | {"Long%act":>9}')
print('-' * 50)
for pair in sorted(results['pair'].unique()):
    sub = results[results['pair'] == pair]
    if len(sub) < 10:
        continue
    acc_p     = accuracy_score(sub['actual_dir'], sub['pred_dir'])
    long_pred = sub['pred_dir'].mean()
    long_act  = sub['actual_dir'].mean()
    print(f'{pair:>8} | {len(sub):>6,} | {acc_p:.4f} | {long_pred:>9.1%} | {long_act:>8.1%}')

## 9. Feature Importance

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 10))
fig.patch.set_facecolor('#080c14')

for ax, model, name in [(axes[0], long_model, 'Long Model'), (axes[1], short_model, 'Short Model')]:
    imp = pd.Series(model.feature_importances_, index=dir_feature_cols).sort_values(ascending=False)
    ax.set_facecolor('#080c14')
    imp.head(25).plot(kind='barh', ax=ax, color='#00d4ff')
    ax.invert_yaxis()
    ax.set_title(f'Top 25 Features — {name}', color='white', fontsize=12)
    ax.tick_params(colors='white')
    for spine in ax.spines.values():
        spine.set_edgecolor('#334455')

plt.tight_layout()
plt.show()

print('\nTop 20 — Long model:')
long_imp = pd.Series(long_model.feature_importances_, index=dir_feature_cols).sort_values(ascending=False)
print(long_imp.head(20).to_string())

print('\nTop 20 — Short model:')
short_imp = pd.Series(short_model.feature_importances_, index=dir_feature_cols).sort_values(ascending=False)
print(short_imp.head(20).to_string())

## 10. Summary

In [ ]:
print('=' * 70)
print('SEPARATE LONG/SHORT DIRECTION MODELS — TRAINING COMPLETE')
print('=' * 70)
print(f'Population:        MFE Q50 > {MFE_THRESHOLD} pips')
print(f'Train samples:     {len(df_dir_train):,}  |  Test samples: {len(df_dir_test):,}')
print(f'Features:          {len(dir_feature_cols)} (features_9 + mfe_q50_oof)')
print()
print(f'Long  model:  OOF acc={acc_long_oof:.4f}   Test acc={acc_long_test:.4f}   AUC={auc_long_test:.4f}   iters={long_iters}')
print(f'Short model:  OOF acc={acc_short_oof:.4f}   Test acc={acc_short_test:.4f}   AUC={auc_short_test:.4f}   iters={short_iters}')
print()
print(f'Combined (argmax):  Test acc={acc_test_combined:.4f}')
print(f'  Long predicted:  {pred_test_combined.mean():.1%}   Actual long rate: {y_long_test.mean():.1%}')
print()
print(f'Inference logic:')
print(f'  p_long  = long_model.predict_proba(X)[:, 1]')
print(f'  p_short = short_model.predict_proba(X)[:, 1]')
print(f'  direction = 1 (long) if p_long >= p_short else -1 (short)')
print(f'  confidence = max(p_long, p_short)')
print()
print(f'Saved: {long_path}')
print(f'Saved: {short_path}')
print('=' * 70)